# 04. Add conditions while preserving the normal reference

A normal-only encoder can serve as a reference, but a five-class classifier may benefit from exposure to more motion patterns. This lesson compares a cumulative condition curriculum with a mixed adaptation control. Every training observation remains inside the training partition.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Keep the question and labels separate

The curriculum adds Parkinsonian, stroke, cerebral-palsy, and myopathic examples after normal pretraining. Their order is an experimental choice, not a claim about disease progression. We retain normal replay at every stage. An adapted representation may become better for classification while becoming a weaker normality reference, so the original normal model is preserved.

Real mode uses actual normal and condition-labeled training windows from the frozen GAVD split. Synthetic mode uses artificial motions whose condition names merely exercise the API. Those synthetic labels must never be interpreted medically.

In [ ]:
display(SVG(filename=str(PROJECT / "images/adaptation.svg")))

In [ ]:
import copy
from worldmodels.data import LABELS
from worldmodels.notebook_support import lesson_data
from worldmodels.model import ModelConfig, SkeletonJEPA
from worldmodels.training import TrainConfig, fit, encode
fixture = lesson_data(n=40, seed=27, conditions=True)
normal_rows = fixture["labels"] == "normal"
reference = SkeletonJEPA(ModelConfig(dim=24, encoder_layers=1))
fit(reference, fixture["x"][normal_rows], fixture["valid"][normal_rows],
    TrainConfig(steps=4, seed=27, regime="normal", run_id="lesson04-normal-v1"),
    labels=fixture["labels"][normal_rows], source_ids=fixture["source_ids"][normal_rows], partitions=["train"]*int(normal_rows.sum()))
reference_weights = {k: v.clone() for k,v in reference.state_dict().items()}
adapted = copy.deepcopy(reference)

## Step 2. Add one label at a time

At each stage, eligible labels are the conditions already introduced plus normal. The sampler reserves normal examples for replay and draws distinct sources. This reduces immediate forgetting of the reference population, but it does not guarantee that earlier representations remain unchanged. Keep each stage's identity and evaluate forgetting using held-out development sources.

In [ ]:
stages = []
for stage, condition in enumerate(LABELS[1:], start=1):
    rows = np.isin(fixture["labels"], LABELS[:stage+1])
    fit(adapted, fixture["x"][rows], fixture["valid"][rows],
        TrainConfig(steps=4, seed=27, regime="adaptation", run_id=f"lesson04-stage-{stage}-v1"),
        labels=fixture["labels"][rows], source_ids=fixture["source_ids"][rows], partitions=["train"]*int(rows.sum()))
    stages.append({"stage": stage, "added": condition, "sources": int(rows.sum()), "parent": adapted.run_metadata["parent_run_id"]})
display(pd.DataFrame(stages))

## Step 3. Verify the reference was preserved

Copying a model before adaptation is a small implementation detail with an important scientific consequence. It allows normal-reference anomaly scores and adapted classification scores to be compared honestly without silently changing what “normal” means.

In [ ]:
assert all(torch.equal(reference_weights[k], reference.state_dict()[k]) for k in reference_weights)
print("Original normal weights are unchanged.")
print("Adapted identity:", adapted.run_metadata["run_id"])

## Step 4. Build a mixed-data control

The mixed model starts from the same normal checkpoint and receives all five labels immediately, with the same total number of adaptation updates. This controls update budget. It does not exactly control exposure to each condition, because curriculum stages have different membership. Report that distinction instead of calling the comparison perfectly matched.

In [ ]:
mixed = copy.deepcopy(reference)
fit(mixed, fixture["x"], fixture["valid"],
    TrainConfig(steps=16, seed=27, regime="adaptation", run_id="lesson04-mixed-v1"),
    labels=fixture["labels"], source_ids=fixture["source_ids"], partitions=["train"]*len(fixture["x"]))
print("Curriculum adaptation updates:", 4*4)
print("Mixed adaptation updates:", mixed.run_metadata["config"]["steps"])

## Step 5. Inspect representation change without claiming accuracy

A change in embeddings shows that training changed the model. It does not establish improvement. Use source-level validation classification, normal-reference error, and representation diagnostics to choose a declared checkpoint. The final test set cannot choose the curriculum order, epoch count, or best-looking visualization.

In [ ]:
representations = {"normal": encode(reference, fixture["x"], fixture["valid"]),
                   "curriculum": encode(adapted, fixture["x"], fixture["valid"]),
                   "mixed": encode(mixed, fixture["x"], fixture["valid"])}
display(pd.DataFrame({name: {"median_dimension_std": float(np.median(z.std(0))), "mean_norm": float(np.linalg.norm(z,axis=1).mean())} for name,z in representations.items()}))

## Step 6. Transfer the protocol to real data

Use the frozen source table from `worldmodels.data.assign_splits`, select rows whose split is `train`, and then select normal labels for Stage A. Use only the training subset again during Stage B. For each source-grouped cross-validation fold, repeat pretraining and adaptation from a fresh model. A checkpoint trained on another fold's test subjects cannot be reused merely because the architecture is identical.

In [ ]:
print("Real cumulative label sets:")
for i in range(1, len(LABELS)):
    print(i, LABELS[:i+1])

In [ ]:
lesson_evidence = persist_lesson("04_progressive_condition_adaptation", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.